# VideoDB Understanding: VLM Guide

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/vlm/vlm-guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Go deeper on VLM analyzers: prompts, inputs, simple schemas, advanced schemas, and tabular inspection.


## 1. Install dependencies

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/video-db/videodb-python.git@feat/add-indexing-v2" python-dotenv pandas

## 2. Connect to VideoDB

In [ ]:
import os
from getpass import getpass

import pandas as pd
from dotenv import load_dotenv
from videodb import connect

load_dotenv()
os.environ["VIDEO_DB_API_KEY"] = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()

print("Connected to VideoDB")
print("Collection:", collection.id)

## 3. Choose a video

By default, this notebook uploads the sample video used in the E2E flow: **Silicon Valley - Gilfoyle is free for hire**. To use an existing video instead, comment the upload line and uncomment the `get_video` lines in the next cell.


In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

collection = conn.get_collection()
video = collection.upload(VIDEO_URL)

# To use an existing video instead, comment the upload line above and uncomment these lines:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


## 4. Small helper to wait and preview output

In [ ]:
def show_vlm_output(understanding, analyzer_name="scene", max_scenes=5):
    understanding.wait_until_complete(timeout=3600, poll_interval=15)

    print("Understanding:", understanding.id)
    print("Status:", understanding.status)
    print()

    for analyzer in understanding.list_analyzers():
        print(analyzer.name, analyzer.type, analyzer.status)

    output = understanding.get_analyzer(analyzer_name).get_output()
    scenes = output.get("scenes", output) if isinstance(output, dict) else output
    scenes = scenes or []

    print("Preview")
    print("=" * 60)
    for scene in scenes[:max_scenes]:
        print(f"{scene.get('start')}s → {scene.get('end')}s")
        print(scene.get("data"))
        print("-" * 60)

    return scenes

## 5. Pick a VLM model

Change this one variable to try another model.

Examples:

```python
VLM_MODEL = "basic"
VLM_MODEL = "pro"
VLM_MODEL = "ultra"
VLM_MODEL = "google/gemini-2.5-flash"
VLM_MODEL = "openai/gpt-4o"
```


In [ ]:
VLM_MODEL = "google/gemini-2.5-flash"

## 6. Starter VLM: frames + prompt

This is the simplest VLM run. VideoDB samples frames from each scene and sends them to the model with your prompt.

In [ ]:
starter_understanding = video.understand(
    analyzers=[
        {
            "type": "vlm",
            "name": "scene",
            "sampling": {"strategy": "uniform", "frame_count": 8},
            "config": {
                "model": VLM_MODEL,
                "prompt": "Describe what is happening in this scene. Mention visible people, objects, actions, and setting.",
            },
        }
    ],
    segmentation={"type": "shot", "threshold": 30},
)

starter_scenes = show_vlm_output(starter_understanding, "scene")

## 7. VLM with inputs: transcript + objects

Use `inputs` when the VLM should also consider other analyzer outputs.

Here, the `scene_with_context` analyzer uses:

- `transcript` from `spoken_words`
- `objects` from `object_detection`


In [ ]:
context_understanding = video.understand(
    analyzers=[
        {"type": "spoken_words", "name": "transcript"},
        {
            "type": "object_detection",
            "name": "objects",
            "sampling": {"strategy": "interval", "every": 1},
            "config": {
                "labels": ["person", "vehicle", "animal", "product", "logo", "text"],
                "confidence_threshold": 0.35,
                "include_bounding_boxes": True,
            },
        },
        {
            "type": "vlm",
            "name": "scene_with_context",
            "inputs": ["transcript", "objects"],
            "sampling": {"strategy": "uniform", "frame_count": 8},
            "config": {
                "model": VLM_MODEL,
                "prompt": "Use the frames, transcript, and detected objects to explain what is happening in this scene.",
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

context_scenes = show_vlm_output(context_understanding, "scene_with_context")

## 8. Simple custom schema

Add `config.schema` when you want JSON fields instead of free-form text.

Simple schema rules:

- The root schema is an object.
- Simple fields are required by default.
- Arrays use Python list syntax, for example `{"objects": ["string"]}`.


In [ ]:
simple_schema = {
    "scene_description": "text",
    "activity": "string",
    "visible_objects": ["string"],
    "confidence": {"type": "number", "min": 0, "max": 1},
}

simple_schema_understanding = video.understand(
    analyzers=[
        {
            "type": "vlm",
            "name": "scene_json",
            "sampling": {"strategy": "uniform", "frame_count": 8},
            "config": {
                "model": VLM_MODEL,
                "prompt": "Analyze the scene and return the requested JSON fields.",
                "schema": simple_schema,
            },
        }
    ],
    segmentation={"type": "shot", "threshold": 30},
)

simple_schema_scenes = show_vlm_output(simple_schema_understanding, "scene_json")

## 9. Advanced custom schema

Use advanced schema fields when you need optional fields, enums, nested objects, arrays of objects, or min/max constraints.

Advanced config keys include:

- `required: False`
- `description`
- `type: "enum"` with `values`
- `type: "object"` with `fields`
- `type: "array"` with `items`


In [ ]:
advanced_schema = {
    "scene": {
        "description": "text",
        "activity": "string",
        "setting": {
            "type": "string",
            "required": False,
            "description": "Visible location or setting, if identifiable.",
        },
        "shot_type": {
            "type": "enum",
            "values": ["wide", "medium", "closeup", "screen", "unknown"],
        },
    },
    "people": {
        "type": "array",
        "min_items": 0,
        "max_items": 5,
        "items": {
            "description": "text",
            "action": "string",
            "confidence": {"type": "number", "min": 0, "max": 1},
        },
    },
    "brands": {
        "type": "array",
        "required": False,
        "items": {
            "name": "string",
            "evidence": "text",
            "confidence": {"type": "number", "min": 0, "max": 1},
        },
    },
}

advanced_schema_understanding = video.understand(
    analyzers=[
        {"type": "spoken_words", "name": "transcript"},
        {
            "type": "vlm",
            "name": "advanced_scene_json",
            "inputs": ["transcript"],
            "sampling": {"strategy": "uniform", "frame_count": 8},
            "config": {
                "model": VLM_MODEL,
                "prompt": "Analyze the sampled frames. Use transcript only when helpful. Return grounded JSON and do not invent details.",
                "schema": advanced_schema,
                "schema_mode": "auto",
                "schema_max_retries": 1,
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

advanced_schema_scenes = show_vlm_output(advanced_schema_understanding, "advanced_scene_json")

## 10. Convert output to a DataFrame

The `data` field contains your VLM response. With schemas, those fields are stable and easier to inspect or index later.

In [ ]:
rows = []

for scene in advanced_schema_scenes:
    row = {
        "scene_id": scene.get("scene_id"),
        "start": scene.get("start"),
        "end": scene.get("end"),
        "data": scene.get("data"),
    }
    rows.append(row)

pd.DataFrame(rows).head()

## 11. Optional cleanup

Uncomment the lines below if you want to delete the runs created by this notebook.

In [ ]:
# starter_understanding.delete()
# context_understanding.delete()
# simple_schema_understanding.delete()
# advanced_schema_understanding.delete()

## Quick reference

### VLM without inputs

```python
{"type": "vlm", "config": {"prompt": "Describe the scene."}}
```

### VLM with inputs

```python
{"type": "vlm", "inputs": ["transcript", "objects"]}
```

### Simple schema

```python
{
    "scene_description": "text",
    "activity": "string",
    "brand_names": ["string"],
    "confidence": {"type": "number", "min": 0, "max": 1},
}
```

### Advanced model/schema config

```python
"config": {
    "model": "google/gemini-2.5-flash",
    "prompt": "Analyze this scene.",
    "schema": advanced_schema,
    "schema_mode": "auto",        # auto | native_required | prompt_only
    "schema_max_retries": 1,
}
```
